In [0]:
%sql
USE CATALOG data_marts;

CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
from pyspark.sql import functions as F

In [0]:
def analisar_null_coluna(tabela):

  nulos = tabela.select([
    F.count(F.when(F.col(c).isNull() , c)).alias(c)
    for c in tabela.columns
  ]).display()

In [0]:
catalogo = "data_marts"

gold_db_name = "gold"
silver_db_name = "silver"

In [0]:
pedidos_total = spark.read.table(f"{catalogo}.{silver_db_name}.fat_pedido_total")
itens_pedidos = spark.read.table(f"{catalogo}.{silver_db_name}.fat_itens_pedidos")
produtos = spark.read.table(f"{catalogo}.{silver_db_name}.dim_produtos")
vendedor = spark.read.table(f"{catalogo}.{silver_db_name}.dim_vendedores")
avaliacoes = spark.read.table(f"{catalogo}.{silver_db_name}.fat_avaliacoes_pedidos")
cotacao = spark.read.table(f"{catalogo}.{silver_db_name}.dim_cotacao_dolar")

In [0]:
df_vendas_completo = (
    itens_pedidos.alias("item")
    .join(produtos.alias("produto"), "id_produto", "inner")
    .join(pedidos_total.alias("pedido"), "id_pedido", "inner")
    .join(cotacao.alias("cotacao"), F.to_date(F.col("pedido.data_pedido")) == F.col("cotacao.data_cotacao"), "left")
)

fat_vendas_comercial = (
    df_vendas_completo
    .groupBy(
        F.year("pedido.data_pedido").alias("ano_venda"),
        F.month("pedido.data_pedido").alias("mes_venda"),
        F.col("produto.categoria_produto")
    )
    .agg(
        F.countDistinct("pedido.id_pedido").alias("total_pedidos"),
        F.count("item.id_item").alias("qtd_itens_vendidos"),
        F.sum("item.preco_BRL").alias("receita_total_brl"),
        
        F.round(F.sum(F.col("item.preco_BRL") / F.col("cotacao.valor_cotacao")), 2).alias("receita_total_usd")
    )
    .select(
        "ano_venda", "mes_venda", "categoria_produto", "total_pedidos", "qtd_itens_vendidos",
        F.round("receita_total_brl", 2).alias("receita_total_brl"),
        F.round("receita_total_usd", 2).alias("receita_total_usd"),
        F.round(F.col("receita_total_brl") / F.col("total_pedidos"), 2).alias("ticket_medio_brl")
    )
    .orderBy("ano_venda", "mes_venda","categoria_produto")
)

fat_vendas_comercial.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db_name}.fat_vendas_comercial")

print("✅ Tabela gold.fat_vendas_comercial criada com sucesso!\n")

In [0]:
tabela_fat_vendas_comercial = spark.table("data_marts.gold.fat_vendas_comercial")
tabela_fat_vendas_comercial.display()

analisar_null_coluna(fat_vendas_comercial)

In [0]:
def gerar_ranking_vendas(catalogo, silver_db, gold_db, nome_tabela_registro, ordem_decrescente):
    pedidos_total = spark.read.table(f"{catalogo}.{silver_db}.fat_pedido_total")
    itens_pedidos = spark.read.table(f"{catalogo}.{silver_db}.fat_itens_pedidos")
    produtos = spark.read.table(f"{catalogo}.{silver_db}.dim_produtos")

    df_base = (
        itens_pedidos.alias("item")
        .join(produtos.alias("produto"), "id_produto", "inner")
        .join(pedidos_total.alias("pedido"), "id_pedido", "inner")
    )

    df_ranking = (
        df_base
        .groupBy(
            F.col("produto.nome_produto"),
            F.col("produto.categoria_produto")
        )
        .agg(
            F.count("item.id_item").alias("quantidade_vendida")
            )
        .orderBy(F.col("quantidade_vendida").desc() if ordem_decrescente else F.col("quantidade_vendida").asc())
        .limit(5)
    )

    caminho_tabela = f"{catalogo}.{gold_db}.{nome_tabela_registro}"
    df_ranking.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(caminho_tabela)
    
    print(f"✅ Tabela {caminho_tabela} criada com sucesso!")
        
    return df_ranking


In [0]:
top_mais_vendido = gerar_ranking_vendas(catalogo, silver_db_name, gold_db_name, "fat_top_5_mais_vendidos", True)

top_menos_vendido = gerar_ranking_vendas(catalogo, silver_db_name, gold_db_name, "fat_top_5_menos_vendidos", False)

top_mais_vendido.display()
top_menos_vendido.display()

In [0]:
df_base = (
    itens_pedidos.alias("item")
    .join(produtos.alias("produto"), "id_produto", "inner")
    .join(pedidos_total.alias("pedido"), "id_pedido", "inner")
    .join(vendedor.alias("vendedor"), "id_vendedor", "inner")
    .join(avaliacoes.alias("avaliacao"), "id_pedido", "inner")
)

fat_avaliacoes_clientes = (
    df_base
    .groupBy(
        F.col("produto.categoria_produto"),
        F.col("vendedor.nome_vendedor"),
        F.col("estado")
    )
    .agg(
        F.count("avaliacao.id_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("avaliacao.nota_avaliacao"), 2).alias("avaliacao_media"),
        
        F.sum(F.when(F.col("avaliacao.nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
        F.sum(F.when(F.col("avaliacao.nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
    )
    .withColumn("percentual_satisfacao", F.round((F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100, 2))
)

fat_avaliacoes_clientes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{gold_db_name}.fat_avaliacoes_clientes")

print("✅ Tabela gold.fat_avaliacoes_clientes criada com sucesso!\n")


In [0]:
tabela_fat_avaliacoes_clientes = spark.table("data_marts.gold.fat_avaliacoes_clientes")
tabela_fat_avaliacoes_clientes.display()

In [0]:
def gerar_ranking_avaliacoes(catalogo, silver_db, gold_db, nome_tabela_registro, ordem_decrescente):
    pedidos_total = spark.read.table(f"{catalogo}.{silver_db_name}.fat_pedido_total")
    itens_pedidos = spark.read.table(f"{catalogo}.{silver_db_name}.fat_itens_pedidos")
    produtos = spark.read.table(f"{catalogo}.{silver_db_name}.dim_produtos")
    avaliacoes = spark.read.table(f"{catalogo}.{silver_db_name}.fat_avaliacoes_pedidos")

    df_base = (
        itens_pedidos.alias("item")
        .join(produtos.alias("produto"), "id_produto", "inner")
        .join(pedidos_total.alias("pedido"), "id_pedido", "inner")
        .join(avaliacoes.alias("avaliacao"), "id_pedido", "inner")
    )

    df_ranking = (
        df_base
        .groupBy(
            F.col("produto.id_produto"),
            F.col("produto.nome_produto"),
            F.col("produto.categoria_produto")
        )
        .agg(
            F.avg("avaliacao.nota_avaliacao").alias("nota_media"),
            F.count("avaliacao.id_avaliacao").alias("volume_avaliacoes")
        )
        .orderBy(
            F.col("nota_media").desc() if ordem_decrescente else F.col("nota_media").asc(),
            F.col("volume_avaliacoes").desc()
        )
        .limit(5)
    )

    caminho_tabela = f"{catalogo}.{gold_db}.{nome_tabela_registro}"
    df_ranking.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(caminho_tabela)
    
    print(f"✅ Tabela {caminho_tabela} criada com sucesso!")
    
    return df_ranking


In [0]:
top_produtos_mais_bem_avaliado = gerar_ranking_avaliacoes(catalogo, silver_db_name, gold_db_name, "fat_top_5_produtos_mais_bem_avaliados", True)

top_produtos_menos_bem_avaliado = gerar_ranking_avaliacoes(catalogo, silver_db_name, gold_db_name, "fat_top_5_produtos_menos_bem_avaliados", False)

top_produtos_mais_bem_avaliado.display()
top_produtos_menos_bem_avaliado.display()


In [0]:
def gerar_ranking_vendedores(catalogo, silver_db, gold_db, nome_tabela_registro, ordem_decrescente):
    pedidos_total = spark.read.table(f"{catalogo}.{silver_db_name}.fat_pedido_total")
    itens_pedidos = spark.read.table(f"{catalogo}.{silver_db_name}.fat_itens_pedidos")
    produtos = spark.read.table(f"{catalogo}.{silver_db_name}.dim_produtos")
    avaliacoes = spark.read.table(f"{catalogo}.{silver_db_name}.fat_avaliacoes_pedidos")
    vendedor = spark.read.table(f"{catalogo}.{silver_db_name}.dim_vendedores")

    df_base = (
        itens_pedidos.alias("item")
        .join(produtos.alias("produto"), "id_produto", "inner")
        .join(pedidos_total.alias("pedido"), "id_pedido", "inner")
        .join(vendedor.alias("vendedor"), "id_vendedor", "inner")
        .join(avaliacoes.alias("avaliacao"), "id_pedido", "inner")
    )

    
    df_ranking = (
        df_base
        .groupBy(
            F.col("vendedor.id_vendedor"),
            F.col("vendedor.nome_vendedor")
        )
        .agg(
            F.round(F.avg("avaliacao.nota_avaliacao"), 2).alias("nota_media"),
            F.count("avaliacao.id_avaliacao").alias("volume_avaliacoes")
        )
        .orderBy(
            F.col("nota_media").desc() if ordem_decrescente else F.col("nota_media").asc(),
            F.col("volume_avaliacoes").desc()
        )
        .limit(5)
    )

    caminho_tabela = f"{catalogo}.{gold_db}.{nome_tabela_registro}"
    df_ranking.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(caminho_tabela)
    
    print(f"✅ Tabela {caminho_tabela} criada com sucesso!")
        
    return df_ranking


In [0]:
top_vendedor_mais_bem_avaliado = gerar_ranking_vendedores(catalogo, silver_db_name, gold_db_name, "fat_top_5_vendedores_mais_bem_avaliados", True)

top_vendedor_menos_bem_avaliado = gerar_ranking_vendedores(catalogo, silver_db_name, gold_db_name, "fat_top_5_vendedores_menos_bem_avaliados", False)

top_vendedor_mais_bem_avaliado.display()
top_vendedor_menos_bem_avaliado.display()
